# Phase 8 — Feature Matrix Construction (v2, updated for Phase 7 v4)

**What changed vs the previous version**
- Removed the two deleted composite KPIs from the priority list (`efficiency`, `cost`) — they no longer exist in Phase 7 v4 output.
- Added the new absolute-RMS effort KPIs as **Priority 1**: `kpi_abs_rms_distal__median` (primary discriminator, p<0.001) and `kpi_abs_rms_mean__median`.
- `kpi_NMD` is intentionally **not** a model feature (it didn't discriminate, p≈0.10) — kept as a descriptive KPI in the report/slides only.
- Added an explicit, **non-destructive ALS_15 exclusion flag** (new cell after the builder), so downstream phases can report results *with and without* the artifact subject.

**Run order:** Cell 1 (build matrix) → Cell 2 (ALS_15 flag) → Cell 3 (inspect).

In [ ]:
# =========================================
# Phase 8 — Feature Matrix Construction
# Input:  processed/phase_07_kpis/subject_kpis.parquet
# Output: phase_08_matrix/{X_matrix, X_scaled, y_labels, meta}.parquet
#         phase_08_matrix/{feature_report, selected_features_info}.csv
# =========================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# --- Paths ---
PROJECT_ROOT   = Path("<PROJECT_ROOT>")
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
SUBJ_KPI_PATH  = PROCESSED_ROOT / "phase_07_kpis" / "subject_kpis.parquet"
PHASE8_DIR     = PROCESSED_ROOT / "phase_08_matrix"
PHASE8_DIR.mkdir(parents=True, exist_ok=True)

# --- Config ---
MIN_COVERAGE    = 0.80   # >=19/23 subjects must have a value
MIN_STD         = 1e-6   # near-zero-variance drop
MAX_FEATURES_FA = 20     # >=5 subjects per variable rule of thumb

print("="*60); print("PHASE 8 — Feature Matrix Construction"); print("="*60)
df_subj = pd.read_parquet(SUBJ_KPI_PATH)
print(f"\nLoaded: {df_subj.shape[0]} subjects x {df_subj.shape[1]} columns")

# --- Step 2: metadata vs KPI ---
META_COLS_CANDIDATES = [
    "subject","group","Group","label","class","exo","EXO","condition","Condition",
    "session","Session","task","task_type","Task",
    "n_trials","n_trials_valid_interval","n_trials_emg_valid","n_trials_has_bictri_pair",
    "rate_valid_interval","rate_emg_valid","rate_has_bictri_pair",
]
meta_cols = [c for c in META_COLS_CANDIDATES if c in df_subj.columns]
print(f"\nMetadata columns ({len(meta_cols)}): {meta_cols}")
df_meta = df_subj[meta_cols].copy() if meta_cols else pd.DataFrame(index=df_subj.index)

# --- Step 3: KPI candidates (overall __median only) ---
kpi_cols_all    = [c for c in df_subj.columns if c.startswith("kpi_")]
kpi_median_cols = [c for c in kpi_cols_all if c.endswith("__median")]
kpi_overall_cols= [c for c in kpi_median_cols if "__taskmed__" not in c]
print(f"\nKPI discovery: total={len(kpi_cols_all)}  overall__median={len(kpi_overall_cols)}")

# --- Step 4: coverage filter ---
df_candidates      = df_subj[kpi_overall_cols].copy()
n_subjects         = len(df_candidates)
coverage           = df_candidates.notna().mean()
high_coverage_cols = coverage[coverage >= MIN_COVERAGE].index.tolist()
print(f"\nCoverage >= {MIN_COVERAGE*100:.0f}%: {len(kpi_overall_cols)} -> {len(high_coverage_cols)}")

# --- Step 5: variance filter ---
df_high_cov      = df_subj[high_coverage_cols].copy()
feature_std      = df_high_cov.std()
nonzero_var_cols = feature_std[feature_std > MIN_STD].index.tolist()
print(f"Variance > {MIN_STD}: {len(high_coverage_cols)} -> {len(nonzero_var_cols)}")

# --- Step 6: feature report ---
rows = []
for col in nonzero_var_cols:
    s = df_subj[col].dropna()
    rows.append({"feature":col,"coverage_pct":round(coverage[col]*100,1),
                 "n_valid":int(coverage[col]*n_subjects),
                 "median":round(s.median(),4) if len(s) else np.nan,
                 "std":round(s.std(),4) if len(s) else np.nan,
                 "min":round(s.min(),4) if len(s) else np.nan,
                 "max":round(s.max(),4) if len(s) else np.nan,
                 "skewness":round(s.skew(),3) if len(s)>2 else np.nan})
df_report = pd.DataFrame(rows).sort_values("coverage_pct", ascending=False)
df_report.to_csv(PHASE8_DIR / "feature_report.csv", index=False)
print(f"\nFeature report saved: {len(df_report)} candidates")

# --- Step 7: clinically-informed priority selection ---
# UPDATED: abs_rms effort KPIs added as Priority 1; efficiency/cost composites removed;
# NMD intentionally excluded as a model feature.
FEATURE_PRIORITY_PATTERNS = [
    ("kpi_abs_rms_distal__median",          "absolute distal EMG RMS (extensor/flexor) - primary effort discriminator", 1),
    ("kpi_abs_rms_mean__median",            "absolute mean EMG RMS across muscles",        1),
    ("kpi_iemg_envn_global_sum__median",    "total EMG activity all muscles",              1),
    ("kpi_coact_overlap_bic_tric__median",  "co-activation biceps-triceps",                1),
    ("kpi_peak_gyro_mag__median",           "peak angular velocity",                       1),
    ("kpi_mean_jerk__median",               "movement smoothness (jerk)",                  1),
    ("kpi_peak_envn_Biceps__median",        "peak activation biceps",                      1),
    ("kpi_peak_envn_Triceps__median",       "peak activation triceps",                     1),
    ("kpi_peak_envn_Deltoid__median",       "peak activation deltoid",                     1),
    ("kpi_iemg_envn_Biceps__median",        "iEMG biceps",                                 2),
    ("kpi_iemg_envn_Triceps__median",       "iEMG triceps",                                2),
    ("kpi_iemg_envn_Deltoid__median",       "iEMG deltoid",                                2),
    ("kpi_duty_Biceps__median",             "duty cycle biceps",                           2),
    ("kpi_duty_Triceps__median",            "duty cycle triceps",                          2),
    ("kpi_mean_gyro_mag__median",           "mean angular velocity",                       2),
    ("kpi_duration_s__median",              "trial duration",                              2),
    ("kpi_peak_envn_Trapezius__median",     "peak activation trapezius",                   2),
    ("kpi_peak_envn_Extensor__median",      "peak activation extensor",                    2),
    ("kpi_peak_envn_Flexor__median",        "peak activation flexor",                      2),
    ("kpi_duty_Deltoid__median",            "duty cycle deltoid",                          3),
]

selected_features = []
for col_name, description, priority in FEATURE_PRIORITY_PATTERNS:
    if col_name in nonzero_var_cols:
        selected_features.append({"feature":col_name,"clinical_meaning":description,"priority":priority})

seen, selected_unique = set(), []
for item in selected_features:
    if item["feature"] not in seen:
        seen.add(item["feature"]); selected_unique.append(item)
selected_unique.sort(key=lambda x: x["priority"])
selected_col_names = [i["feature"] for i in selected_unique]

if len(selected_col_names) > MAX_FEATURES_FA:
    print(f"\nWARN: {len(selected_col_names)} features > FA limit {MAX_FEATURES_FA}; keeping priority 1-2.")
    selected_col_names = [i["feature"] for i in selected_unique if i["priority"] <= 2][:MAX_FEATURES_FA]

print(f"\nFinal features selected: {len(selected_col_names)}")
print(f"\n{'Feature':<48}{'Cov':>6} {'P':>2}  meaning")
print("-"*100)
for item in selected_unique:
    if item["feature"] in selected_col_names:
        cov = coverage.get(item["feature"], np.nan)
        print(f"  {item['feature']:<46}{cov*100:>5.0f}% {item['priority']}   {item['clinical_meaning']}")

# warn if the key discriminator is missing
if "kpi_abs_rms_distal__median" not in selected_col_names:
    print("\n[CHECK] kpi_abs_rms_distal__median NOT selected - is it present in subject_kpis.parquet?")

# --- Step 8: impute (median) ---
X = df_subj[selected_col_names].copy()
n_imp = X.isna().sum().sum()
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())
assert X.isna().sum().sum() == 0
print(f"\nImputation: {n_imp} NaN filled with column median")

# --- Step 9: labels from subject name ---
def map_group_from_name(name: str) -> float:
    n = name.lower().strip()
    if any(k in n for k in ["als","patient","sick","disease"]): return 1.0
    if any(k in n for k in ["healthy","control","hc","normal"]): return 0.0
    return np.nan
subjects = df_subj["subject"].astype(str)
y      = subjects.map(map_group_from_name)
groups = y.map({1.0:"ALS", 0.0:"Healthy"})
if y.isna().sum() > 0:
    print(f"\nERROR: unlabeled subjects: {subjects[y.isna()].tolist()}")
else:
    print(f"\nLabels: {(y==1).sum()} ALS, {(y==0).sum()} Healthy")

# --- Step 10: scale ---
scaler   = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
print(f"\nScaled: mean~{X_scaled.mean().mean():.4f}  std~{X_scaled.std().mean():.4f}")

# --- Step 11: save ---
X.index = subjects.values; X_scaled.index = subjects.values
X.to_parquet(PHASE8_DIR / "X_matrix.parquet", index=True)
X_scaled.to_parquet(PHASE8_DIR / "X_scaled.parquet", index=True)
pd.DataFrame({"subject":subjects.values,"y":y.values,"group_raw":groups.values}).to_parquet(PHASE8_DIR/"y_labels.parquet", index=False)
if "subject" not in df_meta.columns:
    df_meta.insert(0, "subject", subjects.values)
df_meta.to_parquet(PHASE8_DIR / "meta.parquet", index=False)
feat_info = pd.DataFrame(selected_unique)
feat_info = feat_info[feat_info["feature"].isin(selected_col_names)].merge(
    df_report[["feature","coverage_pct","std","skewness"]], on="feature", how="left")
feat_info.to_csv(PHASE8_DIR / "selected_features_info.csv", index=False)

print(f"\n{'='*60}\nPhase 8 complete -> {PHASE8_DIR}\n{'='*60}")
print(f"   X: {X.shape[0]} subjects x {X.shape[1]} features")
print(f"   ALS={int((y==1).sum())}  Healthy={int((y==0).sum())}  NaN remaining={X.isna().sum().sum()>0}")
print("\n   Top 5 by variance:")
for fn, fs in X.std().sort_values(ascending=False).head(5).items():
    print(f"      {fn:<48} std={fs:.4f}")

In [ ]:
# ===== ALS_15 artifact flag (explicit, logged, non-destructive) =====
# ALS_Subject_15 has a confirmed EMG device artifact (extensor RMS ~0.957 V).
# We do NOT delete it: we flag it so every downstream analysis can include or
# exclude it, and we can report the finding "with vs without" the artifact subject.
import pandas as pd
from pathlib import Path

PHASE8_DIR = Path("<PROJECT_ROOT>/data/processed/phase_08_matrix")
ARTIFACT_SUBJECTS = ["ALS_Subject_15"]   # reason: EMG device artifact

meta = pd.read_parquet(PHASE8_DIR / "meta.parquet")
if "subject" not in meta.columns:
    raise KeyError("meta.parquet has no 'subject' column - check Cell 1 save step.")
meta["excluded"] = meta["subject"].isin(ARTIFACT_SUBJECTS)
meta["exclude_reason"] = meta["subject"].map(
    {s: "EMG device artifact (extensor RMS ~0.957 V)" for s in ARTIFACT_SUBJECTS}).fillna("")
meta.to_parquet(PHASE8_DIR / "meta.parquet", index=False)

print(f"Flagged excluded: {meta.loc[meta['excluded'],'subject'].tolist()}")
print(f"Analysis set (excluded=False): {(~meta['excluded']).sum()} of {len(meta)} subjects")
print("\nDownstream usage:")
print("  full set     : all subjects")
print("  analysis set : meta.loc[~meta['excluded'], 'subject']")

In [ ]:
# ===== Inspect the built matrix =====
import pandas as pd
from pathlib import Path
PHASE8_DIR = Path("<PROJECT_ROOT>/data/processed/phase_08_matrix")

X    = pd.read_parquet(PHASE8_DIR / "X_matrix.parquet")
y    = pd.read_parquet(PHASE8_DIR / "y_labels.parquet")
info = pd.read_csv(PHASE8_DIR / "selected_features_info.csv")

print("X matrix:", X.shape)
display(info)
print("\nSelected features:")
for c in X.columns: print("  ", c)
print("\nabs_rms_distal present?", "kpi_abs_rms_distal__median" in X.columns)

In [ ]:
"""
Phase 8 — ALS vs HC overlaid distributions for the 20 selected KPIs.

Built to match the Phase 8 output structure:
    X_matrix.parquet            -> 20 KPI columns, subject as the index
    y_labels.parquet            -> labels (ALS = 1, healthy = 0), aligned to X
    selected_features_info.csv  -> feature metadata (priority, coverage, skew)

For each KPI it overlays the ALS and HC distributions on the same axes, marks
each group's median, and annotates the Mann-Whitney p-value and skewness — so
you can SEE whether the two groups separate, with the numbers right there.

Uses the RAW (unscaled) matrix on purpose: StandardScaler would flatten the
real shape and units you want to inspect here.
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ----------------------------------------------------------------------
# 1. CONFIG
# ----------------------------------------------------------------------
PHASE8_DIR = Path("<PROJECT_ROOT>/data/processed/phase_08_matrix")
FIG_DIR    = Path("<PROJECT_ROOT>/figures/phase08_distributions")
EXCLUDE_ARTIFACT = True                 # drop the flagged artifact for the amplitude view
ARTIFACT_ID      = "ALS_Subject_15"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------
# 2. LOAD  (same files as your last Phase 8 cell)
# ----------------------------------------------------------------------
X    = pd.read_parquet(PHASE8_DIR / "X_matrix.parquet")
y    = pd.read_parquet(PHASE8_DIR / "y_labels.parquet")
info = pd.read_csv(PHASE8_DIR / "selected_features_info.csv")
print("X matrix:", X.shape)

# --- align labels to X and build a readable group column ---
# y is a 1-column frame (ALS=1, HC=0); squeeze it to a Series aligned on the index
y_series = y.iloc[:, 0] if isinstance(y, pd.DataFrame) else y
y_series = y_series.reindex(X.index)            # ensure same order as X
group = np.where(y_series.values == 1, "ALS", "HC")

data = X.copy()
data["group"] = group
data["subject"] = X.index.astype(str)           # subject id is the index

# --- optionally drop the flagged artifact subject ---
if EXCLUDE_ARTIFACT and (data["subject"] == ARTIFACT_ID).any():
    data = data[data["subject"] != ARTIFACT_ID].copy()
    print(f"Excluded {ARTIFACT_ID}  ->  {len(data)} subjects "
          f"({(data['group']=='ALS').sum()} ALS, {(data['group']=='HC').sum()} HC)")
else:
    print(f"{len(data)} subjects "
          f"({(data['group']=='ALS').sum()} ALS, {(data['group']=='HC').sum()} HC)")

# ----------------------------------------------------------------------
# 3. KPI COLUMNS  (the 20 selected features)
# ----------------------------------------------------------------------
kpi_cols = [c for c in X.columns]
print(f"Plotting {len(kpi_cols)} KPIs.")

def nice(name):
    return name.replace("kpi_", "").replace("__median", "").replace("_", " ")

# ----------------------------------------------------------------------
# 4. OVERLAID ALS vs HC PANEL PER KPI
# ----------------------------------------------------------------------
n      = len(kpi_cols)
ncols  = 4
nrows  = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows))
axes = axes.flatten()

C_ALS, C_HC = "#d1495b", "#3a7ca5"

summary_rows = []
for i, col in enumerate(kpi_cols):
    ax = axes[i]
    als = data.loc[data["group"] == "ALS", col].dropna().values
    hc  = data.loc[data["group"] == "HC",  col].dropna().values

    bins = np.histogram_bin_edges(np.concatenate([als, hc]), bins=8)
    ax.hist(hc,  bins=bins, alpha=0.55, color=C_HC,  label="HC",  edgecolor="white")
    ax.hist(als, bins=bins, alpha=0.55, color=C_ALS, label="ALS", edgecolor="white")
    ax.axvline(np.median(hc),  color=C_HC,  lw=2, ls="--")
    ax.axvline(np.median(als), color=C_ALS, lw=2, ls="--")

    try:
        _, p = stats.mannwhitneyu(als, hc, alternative="two-sided")
    except ValueError:
        p = np.nan
    sk   = stats.skew(np.concatenate([als, hc]))
    star = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

    ax.set_title(f"{nice(col)}\np={p:.3f} {star} | skew={sk:.2f}", fontsize=9)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

    summary_rows.append({
        "kpi": col,
        "ALS_median": np.median(als),
        "HC_median":  np.median(hc),
        "ratio_ALS_over_HC": (np.median(als) / np.median(hc)
                              if np.median(hc) != 0 else np.nan),
        "mannwhitney_p": p,
        "skewness": sk,
    })

for j in range(n, len(axes)):
    axes[j].axis("off")

fig.suptitle("ALS vs HC distributions — 20 selected KPIs (Phase 8)\n"
             "dashed lines = group medians", fontsize=13, y=1.005)
fig.tight_layout()
out_grid = FIG_DIR / "kpi_distributions_ALS_vs_HC.png"
fig.savefig(out_grid, dpi=150, bbox_inches="tight")
print(f"saved grid -> {out_grid}")
plt.show()

# ----------------------------------------------------------------------
# 5. SUMMARY TABLE — sorted by separation (smallest p first)
# ----------------------------------------------------------------------
summary = pd.DataFrame(summary_rows).sort_values("mannwhitney_p")
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("\n=== Per-KPI summary (most-separating first) ===")
print(summary.to_string(index=False))
summary.to_csv(FIG_DIR / "kpi_distribution_summary.csv", index=False)
print(f"\nsaved summary -> {FIG_DIR / 'kpi_distribution_summary.csv'}")

In [ ]:
import pandas as pd
from pathlib import Path
PHASE8_DIR = Path("<PROJECT_ROOT>/data/processed/phase_08_matrix")

X = pd.read_parquet(PHASE8_DIR / "X_matrix.parquet")
y = pd.read_parquet(PHASE8_DIR / "y_labels.parquet")

print("X index name:", X.index.name)
print("X index sample:", list(X.index[:5]))
print("\ny shape:", y.shape)
print("y columns:", list(y.columns))
print("y index name:", y.index.name)
print("y index sample:", list(y.index[:5]))
print("\ny full:\n", y.head(10))
print("\nlabel value counts:\n", y.iloc[:,0].value_counts(dropna=False))

In [ ]:
# if X and y are already in the same row order (they are, from Phase 8):
y_series = y.iloc[:, 0].reset_index(drop=True)
group = np.where(y_series.values == 1, "ALS", "HC")
# then attach by position, not by index:
data = X.reset_index().rename(columns={X.index.name or "index": "subject"})
data["group"] = group

In [ ]:
"""
Phase 8 — ALS vs Healthy overlaid distributions for the 20 selected KPIs.

Matched to the real Phase 8 file structure:
    X_matrix.parquet  -> 20 KPI columns; subject ID is the (unnamed) index
                         e.g. 'ALS_Subject_1', 'Healthy_Subject_1', ...
    y_labels.parquet  -> columns ['subject', 'y', 'group_raw'];
                         integer index; 'group_raw' is 'ALS' / 'Healthy'

Labels are joined to X by MERGING ON THE SUBJECT ID (string-to-string),
so row order / index mismatches can never mislabel anyone.

Uses the RAW (unscaled) matrix on purpose: StandardScaler would flatten the
real shape and units we want to inspect here.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ----------------------------------------------------------------------
# 1. CONFIG
# ----------------------------------------------------------------------
PHASE8_DIR = Path("<PROJECT_ROOT>/data/processed/phase_08_matrix")
FIG_DIR    = Path("<PROJECT_ROOT>/figures/phase08_distributions")
EXCLUDE_ARTIFACT = True
ARTIFACT_ID      = "ALS_Subject_15"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------
# 2. LOAD
# ----------------------------------------------------------------------
X = pd.read_parquet(PHASE8_DIR / "X_matrix.parquet")
y = pd.read_parquet(PHASE8_DIR / "y_labels.parquet")
print("X matrix:", X.shape)

kpi_cols = list(X.columns)                      # the 20 selected features

# --- bring the subject id out of X's index into a column ---
Xr = X.reset_index().rename(columns={"index": "subject"})
if "subject" not in Xr.columns:                 # if the index had no name
    Xr = Xr.rename(columns={Xr.columns[0]: "subject"})

# --- MERGE on subject id (robust to ordering) ---
data = Xr.merge(y[["subject", "group_raw"]], on="subject", how="left")
data = data.rename(columns={"group_raw": "group"})

# sanity check: did every subject get a label?
n_missing = data["group"].isna().sum()
print("subjects with no label after merge:", n_missing)
print("group counts:\n", data["group"].value_counts())

# --- optionally drop the flagged artifact subject ---
if EXCLUDE_ARTIFACT and (data["subject"] == ARTIFACT_ID).any():
    data = data[data["subject"] != ARTIFACT_ID].copy()
    print(f"\nExcluded {ARTIFACT_ID} -> {len(data)} subjects")
    print("group counts after exclusion:\n", data["group"].value_counts())

# the two group names present in your data
G_ALS, G_HC = "ALS", "Healthy"

def nice(name):
    return name.replace("kpi_", "").replace("__median", "").replace("_", " ")

# ----------------------------------------------------------------------
# 3. OVERLAID PANEL PER KPI
# ----------------------------------------------------------------------
n     = len(kpi_cols)
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows))
axes = axes.flatten()
C_ALS, C_HC = "#d1495b", "#3a7ca5"

summary_rows = []
for i, col in enumerate(kpi_cols):
    ax  = axes[i]
    als = data.loc[data["group"] == G_ALS, col].dropna().values
    hc  = data.loc[data["group"] == G_HC,  col].dropna().values

    bins = np.histogram_bin_edges(np.concatenate([als, hc]), bins=8)
    ax.hist(hc,  bins=bins, alpha=0.55, color=C_HC,  label="Healthy", edgecolor="white")
    ax.hist(als, bins=bins, alpha=0.55, color=C_ALS, label="ALS",     edgecolor="white")
    ax.axvline(np.median(hc),  color=C_HC,  lw=2, ls="--")
    ax.axvline(np.median(als), color=C_ALS, lw=2, ls="--")

    try:
        _, p = stats.mannwhitneyu(als, hc, alternative="two-sided")
    except ValueError:
        p = np.nan
    sk   = stats.skew(np.concatenate([als, hc]))
    star = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

    ax.set_title(f"{nice(col)}\np={p:.3f} {star} | skew={sk:.2f}", fontsize=9)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

    summary_rows.append({
        "kpi": col,
        "ALS_median": np.median(als),
        "HC_median":  np.median(hc),
        "ratio_ALS_over_HC": (np.median(als) / np.median(hc)
                              if np.median(hc) != 0 else np.nan),
        "mannwhitney_p": p,
        "skewness": sk,
    })

for j in range(n, len(axes)):
    axes[j].axis("off")

fig.suptitle("ALS vs Healthy distributions — 20 selected KPIs (Phase 8)\n"
             "dashed lines = group medians", fontsize=13, y=1.005)
fig.tight_layout()
out_grid = FIG_DIR / "kpi_distributions_ALS_vs_HC.png"
fig.savefig(out_grid, dpi=150, bbox_inches="tight")
print(f"\nsaved grid -> {out_grid}")
plt.show()

# ----------------------------------------------------------------------
# 4. SUMMARY TABLE — most-separating first
# ----------------------------------------------------------------------
summary = pd.DataFrame(summary_rows).sort_values("mannwhitney_p")
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("\n=== Per-KPI summary (most-separating first) ===")
print(summary.to_string(index=False))
summary.to_csv(FIG_DIR / "kpi_distribution_summary.csv", index=False)
print(f"\nsaved summary -> {FIG_DIR / 'kpi_distribution_summary.csv'}")